# 02 — Train YOLOv11

Trains YOLOv11-Medium on the BDD100K clear-weather training split.
Checkpoints are saved every 10 epochs to `checkpoints/yolov11/`.

**Prerequisites:** run `01_setup_and_data.ipynb` first.

In [ ]:
from pathlib import Path
import torch
from dotenv import load_dotenv
from src.train_utils import setup_logging

DRIVE_ROOT    = Path('/content/drive/MyDrive/FON/master_rad/bdd100k_data')
CONFIGS_DIR   = Path('/content/data_prepared/configs')
CHECKPOINTS   = DRIVE_ROOT / 'checkpoints'

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# Load model
model = YOLO('yolo11m.pt')  # downloads COCO pretrained weights on first run

In [ ]:
results = model.train(
    data=str(CONFIGS_DIR / 'dataset.yaml'),
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    device=DEVICE,
    project=str(CHECKPOINTS / 'yolov11'),
    name='run',
    save_period=10,
    exist_ok=True,
    plots=True,
)
print('Training complete.')
print('Best mAP@50:', results.results_dict.get('metrics/mAP50(B)'))

In [ ]:
import json

summary = {
    'model': 'yolov11',
    'best_map50': results.results_dict.get('metrics/mAP50(B)'),
    'best_map50_95': results.results_dict.get('metrics/mAP50-95(B)'),
    'epochs': 100,
}
out = CHECKPOINTS / 'yolov11' / 'training_summary.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2))
print('Summary saved to', out)